In [1]:
import pandas as pd
import xarray as xr
import numpy as np
import gcsfs
import os
import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv('https://storage.googleapis.com/cmip6/cmip6-zarr-consolidated-stores.csv')
gcs = gcsfs.GCSFileSystem(token='anon')  # Get access to Google Cloud

In [3]:
# For EC-Earth3
# Obtain historical, ssp370, and ssp585 runs ; specify 'tas' or 'psl' here
hist_data = df.query("table_id == 'Amon' & variable_id == 'psl' & experiment_id == 'historical' & source_id == 'EC-Earth3'")
ssp370_data = df.query("table_id == 'Amon' & variable_id == 'psl' & experiment_id == 'ssp370' & source_id == 'EC-Earth3'")
ssp585_data = df.query("table_id == 'Amon' & variable_id == 'psl' & experiment_id == 'ssp585' & source_id == 'EC-Earth3'")

# Stack the historical and scenarios to run from 1850-2047
ssp370_merge = pd.merge(hist_data, ssp370_data, how='inner', on=['institution_id', 'source_id', 'member_id', 'table_id', 'version'])
ssp585_merge = pd.merge(hist_data, ssp585_data, how='inner', on=['institution_id', 'source_id', 'member_id', 'table_id', 'version'])

# Make sure they all have the same physics and forcings
df_hist = hist_data[hist_data['member_id'].str.contains("i1p1f1")]
df_ssp370 = ssp370_merge[ssp370_merge['member_id'].str.contains("i1p1f1")]
df_ssp585 = ssp585_merge[ssp585_merge['member_id'].str.contains("i1p1f1")]

models = df_hist['source_id'].unique()
print('Number of models:', len(models))
print(models)

Number of models: 0
[]


In [4]:
# For all
# Obtain historical, ssp370, and ssp585 runs ; specify 'tas' or 'psl' here
hist_data = df.query("table_id == 'Amon' & variable_id == 'psl' & experiment_id == 'historical'")
ssp370_data = df.query("table_id == 'Amon' & variable_id == 'psl' & experiment_id == 'ssp370'")
ssp585_data = df.query("table_id == 'Amon' & variable_id == 'psl' & experiment_id == 'ssp585'")

# Stack the historical and scenarios to run from 1850-2047
ssp370_merge = pd.merge(hist_data, ssp370_data, how='inner', on=['institution_id', 'source_id', 'member_id', 'table_id'])
ssp585_merge = pd.merge(hist_data, ssp585_data, how='inner', on=['institution_id', 'source_id', 'member_id', 'table_id'])

# Make sure they all have the same physics and forcings
df_hist = hist_data[hist_data['member_id'].str.contains("i1p1f1")]
df_ssp370 = ssp370_merge[ssp370_merge['member_id'].str.contains("i1p1f1")]
df_ssp585 = ssp585_merge[ssp585_merge['member_id'].str.contains("i1p1f1")]

models = df_hist['source_id'].unique()
print('Number of models:', len(models))
print(models)

models = np.delete(models, 21)
print('Number of models:', len(models))
print(models)

Number of models: 55
['GFDL-ESM4' 'GFDL-CM4' 'IPSL-CM6A-LR' 'GISS-E2-1-G' 'BCC-CSM2-MR'
 'MIROC6' 'BCC-ESM1' 'AWI-CM-1-1-MR' 'MRI-ESM2-0' 'CESM2-WACCM' 'CESM2'
 'SAM0-UNICON' 'GISS-E2-1-H' 'CanESM5' 'INM-CM4-8' 'INM-CM5-0'
 'MPI-ESM-1-2-HAM' 'NESM3' 'CAMS-CSM1-0' 'MPI-ESM1-2-LR' 'MPI-ESM1-2-HR'
 'E3SM-1-0' 'MCM-UA-1-0' 'GISS-E2-1-G-CC' 'NorESM2-LM' 'FGOALS-g3'
 'KACE-1-0-G' 'NorCPM1' 'FGOALS-f3-L' 'ACCESS-CM2' 'NorESM2-MM'
 'FIO-ESM-2-0' 'ACCESS-ESM1-5' 'CESM2-WACCM-FV2' 'GISS-E2-2-H' 'CESM2-FV2'
 'E3SM-1-1' 'IITM-ESM' 'EC-Earth3-Veg' 'EC-Earth3' 'AWI-ESM-1-1-LR'
 'EC-Earth3-Veg-LR' 'CIESM' 'CMCC-CM2-SR5' 'E3SM-1-1-ECA' 'TaiESM1'
 'EC-Earth3-AerChem' 'IPSL-CM5A2-INCA' 'CMCC-CM2-HR4' 'CAS-ESM2-0'
 'EC-Earth3-CC' 'CMCC-ESM2' 'ICON-ESM-LR' 'IPSL-CM6A-LR-INCA' 'KIOST-ESM']
Number of models: 54
['GFDL-ESM4' 'GFDL-CM4' 'IPSL-CM6A-LR' 'GISS-E2-1-G' 'BCC-CSM2-MR'
 'MIROC6' 'BCC-ESM1' 'AWI-CM-1-1-MR' 'MRI-ESM2-0' 'CESM2-WACCM' 'CESM2'
 'SAM0-UNICON' 'GISS-E2-1-H' 'CanESM5' 'INM-CM4-8' 'INM-CM5-

In [5]:
# HISTORICAL
storage_path = '/glade/derecho/scratch/skygale/rawdata/hist_psl_month/'

count = 0
for source_id in models:

    print(count, source_id)
    count += 1

    # Get historical runs from current model
    hist_runs = df_hist[df_hist['source_id'] == source_id]

    for i in range(len(hist_runs)):

        # Download historical runs
        zstore = hist_runs.zstore.values[i]

        # Create a mutable-mapping-style interface to the store
        mapper = gcs.get_mapper(zstore)

        # Open it using xarray and zarr
        ds = xr.open_zarr(mapper, consolidated=True, decode_times=False)

        # Create path to appropriate storage directory
        dir = os.path.join(storage_path, source_id)

        # Find unique realization
        realization = hist_runs.iloc[i].member_id

        # Add data if directory does exist
        if os.path.exists(dir):
            print('Directory already exists for', source_id)
            ds.to_netcdf(dir + '/' + realization)

        # Make directory if it doesn't already exist
        else:
            print('Making directory for', source_id)
            os.mkdir(dir)
            ds.to_netcdf(dir + '/' + realization)

0 EC-Earth3
Directory already exists for EC-Earth3
Directory already exists for EC-Earth3
Directory already exists for EC-Earth3
Directory already exists for EC-Earth3
Directory already exists for EC-Earth3
Directory already exists for EC-Earth3
Directory already exists for EC-Earth3
Directory already exists for EC-Earth3
Directory already exists for EC-Earth3
Directory already exists for EC-Earth3
Directory already exists for EC-Earth3
Directory already exists for EC-Earth3
Directory already exists for EC-Earth3
Directory already exists for EC-Earth3
Directory already exists for EC-Earth3
Directory already exists for EC-Earth3
Directory already exists for EC-Earth3
Directory already exists for EC-Earth3
Directory already exists for EC-Earth3
Directory already exists for EC-Earth3
Directory already exists for EC-Earth3
Directory already exists for EC-Earth3
Directory already exists for EC-Earth3
Directory already exists for EC-Earth3
Directory already exists for EC-Earth3
Directory alr

In [6]:
# SSP
storage_path = '/glade/derecho/scratch/skygale/rawdata/ssp_sat_month/'

count = 0
for source_id in df_ssp585['source_id'].unique():

    print(count, source_id)
    count += 1

    # Get SSP runs from current model
    ssp370 = df_ssp370[df_ssp370['source_id'] == source_id]
    ssp585 = df_ssp585[df_ssp585['source_id'] == source_id]

    # Get larger scenario EM
    if len(ssp370) >= len(ssp585):
        chosen_ssp = ssp370
    elif len(ssp585) > len(ssp370):
        chosen_ssp = ssp585

    # Loop for LENS with more/less than 10 EM
    if len(chosen_ssp) >= 10:

        for i in range(len(chosen_ssp)):

            # Download SSP option
            zstore = chosen_ssp.zstore_y.values[i]

            # Create a mutable-mapping-style interface to the store
            mapper = gcs.get_mapper(zstore)

            # Open it using xarray and zarr
            ds = xr.open_zarr(mapper, consolidated=True, decode_times=False)

            # Create path to appropriate storage directory
            dir = os.path.join(storage_path, source_id)

            # Find unique realization
            realization = chosen_ssp.iloc[i].member_id

            # Add data if directory does exist
            if os.path.exists(dir):
                print('Directory already exists for', source_id)
                ds.to_netcdf(dir + '/' + realization)

            # Make directory if it doesn't already exist
            else:
                print('Making directory for', source_id)
                os.mkdir(dir)
                ds.to_netcdf(dir + '/' + realization)

0 EC-Earth3
Directory already exists for EC-Earth3
Directory already exists for EC-Earth3
Directory already exists for EC-Earth3
Directory already exists for EC-Earth3
Directory already exists for EC-Earth3
Directory already exists for EC-Earth3
Directory already exists for EC-Earth3
Directory already exists for EC-Earth3
Directory already exists for EC-Earth3
Directory already exists for EC-Earth3
Directory already exists for EC-Earth3
Directory already exists for EC-Earth3
Directory already exists for EC-Earth3
Directory already exists for EC-Earth3
Directory already exists for EC-Earth3
Directory already exists for EC-Earth3
Directory already exists for EC-Earth3
Directory already exists for EC-Earth3
Directory already exists for EC-Earth3
Directory already exists for EC-Earth3
Directory already exists for EC-Earth3
Directory already exists for EC-Earth3
Directory already exists for EC-Earth3
Directory already exists for EC-Earth3
Directory already exists for EC-Earth3
Directory alr